In [ ]:
import os
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

import ee
import geemap

In [ ]:
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
# ---- Project paths (matches your repo layout) -------------------------------
PROJECT_ROOT = Path.home() / "Documents" / "Projects" / "Fire_emission_rivers"
DATA_DIR     = PROJECT_ROOT / "data" / "Godavari_River"
OUT_DIR      = PROJECT_ROOT / "outputs" / "Godavari_River"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Basin inputs -------------------------------------------------------------
BASIN_GPKG = DATA_DIR / "Godavari_Basin.gpkg"      # preferred
BASIN_SHP  = DATA_DIR / "Godavari_Shape.shp"        # fallback

# ---- Study period ---------------------------------------------------------------
START_YEAR = 2019
END_YEAR   = 2024   # inclusive


In [ ]:
# Load basin boundary

def load_basin_boundary():
    if BASIN_GPKG.exists():
        gdf = gpd.read_file(BASIN_GPKG)
    elif BASIN_SHP.exists():
        gdf = gpd.read_file(BASIN_SHP)
    else:
        raise FileNotFoundError(f"No basin boundary found at {BASIN_GPKG} or {BASIN_SHP}")

    gdf = gdf.to_crs(epsg=4326)
    gdf["dissolve_key"] = "Godavari"
    basin_gdf = gdf.dissolve(by="dissolve_key").reset_index(drop=True)
    basin_gdf["basin_name"] = "Godavari"
    return basin_gdf

basin_gdf = load_basin_boundary()
basin_bbox = tuple(basin_gdf.total_bounds)  # (minx, miny, maxx, maxy)
print("Basin bbox:", basin_bbox)
basin_gdf.plot(edgecolor="black", facecolor="tan", alpha=0.5, figsize=(6, 6))
plt.title("Godavari Basin Boundary")
plt.show()

### FIRMS Fire Activity Dataset

In [ ]:
# ---- NASA FIRMS API -------------------------------------------------------------
# Get a free MAP KEY at: https://firms.modaps.eosdis.nasa.gov/api/map_key/
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current working directory (your project root)
FIRMS_MAP_KEY = os.environ["FIRMS_MAP_KEY"]

if not FIRMS_MAP_KEY:
    raise ValueError("FIRMS_MAP_KEY not found, check your .env file")

# Archive ("Standard Processing") sources — combine MODIS + VIIRS for full FRP coverage
FIRMS_SOURCES = ["MODIS_SP", "VIIRS_SNPP_SP", "VIIRS_NOAA20_SP"]